# Inception 模块：多尺度并行卷积

Inception 的核心思想：在同一层里并行使用不同尺度的卷积，让模型同时观察局部细节和更大范围的上下文。

## 学习目标
- 理解 1×1、3×3、5×5 卷积在感受野上的差异。
- 看懂多分支结构如何并行提取特征后再拼接。
- 掌握 `torch.cat` 在通道维度合并特征图的用法。

## 导入依赖

这里只需要 PyTorch 的网络层和函数接口。Inception 模块本身不依赖具体数据集，可以先专注结构设计。

In [2]:
from torch import nn
from torch.utils.data import DataLoader
from torch.nn import functional as F
from tqdm import tqdm

import torch
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## InceptionBlock 结构解析

每个 InceptionBlock 将同一输入送入四条并行分支：

| 分支 | 操作 | 目的 |
|------|------|------|
| 1×1 卷积 | 通道变换 | 调整通道数，降低计算量 |
| 3×3 卷积 | 空间特征提取 | 捕捉中等范围模式 |
| 5×5 卷积 | 大感受野提取 | 捕捉更大范围上下文 |
| 最大池化 | 下采样 | 保留局部显著特征 |

各分支输出沿**通道维**拼接（`torch.cat(dim=1)`），让下一层能同时接收不同尺度的信息。

In [3]:
# Inception模块的核心思想是通过并行的卷积层来提取不同尺度的特征，从而增强模型的表达能力。
# Inception模块
class InceptionBlock(nn.Module):
    def __init__(self, input_channels, output_channels_list):
        super(InceptionBlock, self).__init__()
        self.conv1_1 = nn.Conv2d(
            in_channels=input_channels,
            out_channels=output_channels_list[0],
            kernel_size=1,
            padding="same"
        )
        self.conv3_3 = nn.Conv2d(
            in_channels=input_channels,
            out_channels=output_channels_list[1],
            kernel_size=3,
            padding="same"
        )
        self.conv5_5 = nn.Conv2d(
            in_channels=input_channels,
            out_channels=output_channels_list[2],
            kernel_size=5,
            padding="same"
        )
        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        out1 = F.relu(self.conv1_1(x))
        out2 = F.relu(self.conv3_3(x))
        out3 = F.relu(self.conv5_5(x))
        max_pooling = self.max_pool(x)  # 进行最大池化，缩小特征图的尺寸

        # 为了将池化后的特征图与其他卷积层的输出进行拼接，需要对池化后的特征图进行填充，使其尺寸与输入相同
        max_pooling_shape = max_pooling.shape[1:]   # 通道，高度，宽度
        input_shape = x.shape[1:]  # 输入的通道，高度，宽度

        # 输入尺寸减去池化后尺寸的一半，得到需要填充的宽度
        width_padding = (input_shape[-2] - max_pooling_shape[-2]) // 2

        # 输入尺寸减去池化后尺寸的一半，得到需要填充的高度
        height_padding = (input_shape[-1] - max_pooling_shape[-1]) // 2

        padded_pooling = F.pad(  # 对池化后的特征图进行填充，使其尺寸与输入相同
            # 填充的顺序是 (左, 右, 上, 下)
            max_pooling, (width_padding, width_padding, height_padding, height_padding))

        concat = torch.cat([out1, out2, out3, padded_pooling], dim=1)

        return concat


class InceptionNet(nn.Module):
    def __init__(self, num_classes=10):
        super(InceptionNet, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, "same"),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            InceptionBlock(32, [16, 16, 16]),   # 32 + 16 + 16 + 16 = 80
            InceptionBlock(80, [16, 16, 16]),   # 80 + 16 + 16 + 16 = 128
            nn.MaxPool2d(2, 2),
            InceptionBlock(128, [16, 16, 16]),  # 128 + 16 + 16 + 16 = 176
            InceptionBlock(176, [16, 16, 16]),  # 176 + 16 + 16 + 16 = 224
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(224 * 4 * 4, num_classes)
        )

    def forward(self, x):
        return self.model(x)
    
print("Total parameters:", sum(p.numel() for p in InceptionNet().parameters()))

Total parameters: 269898
